# Sanitized Sandbox API Notebook

Outputs, execution metadata, and pasted authorization code were removed. Add sandbox credentials locally before running.


# Open Banking Lab — TrueLayer Sandbox API
### Full pipeline: OAuth2 → AIS data extraction → analysis

This notebook walks through the complete TrueLayer sandbox integration:
authenticate via OAuth2, pull account/balance/transaction/card data, clean it,
and run a set of increasingly analytical exercises drawn from real payment analytics work.

---

### Prerequisites

| Requirement | Where to find it |
|---|---|
| TrueLayer Developer account | [console.truelayer.com](https://console.truelayer.com) |
| Sandbox app `client_id` | TrueLayer Console → Your App |
| Sandbox app `client_secret` | TrueLayer Console → Your App |
| Redirect URI | Set to `https://console.truelayer.com/redirect-page` |

> **Sandbox vs Production:** All calls use `truelayer-sandbox.com`. No real bank credentials — you log in with TrueLayer's mock bank accounts.

---

### API Base URLs

```
Auth server : https://auth.truelayer-sandbox.com
Data API    : https://api.truelayer-sandbox.com/data/v1
```


## Imports and configuration

In [ ]:
!pip install requests pandas --quiet

import requests
import pandas as pd
import json
import os
from urllib.parse import urlencode, urlparse, parse_qs
from datetime import datetime

# ── Paste your sandbox credentials here ──────────────────────────────────────
CLIENT_ID     = ""
CLIENT_SECRET = ""
REDIRECT_URI  = "https://console.truelayer.com/redirect-page"

AUTH_BASE = "https://auth.truelayer-sandbox.com"
DATA_BASE = "https://api.truelayer-sandbox.com/data/v1"


---
## Build the authorisation URL

The OAuth2 flow starts by sending the user to TrueLayer's **consent screen**.

### Key parameters

| Parameter | Value | Purpose |
|---|---|---|
| `response_type` | `code` | Request an auth code, not a direct token |
| `client_id` | your app ID | Identifies your application |
| `scope` | `accounts balance transactions cards offline_access` | Data you are requesting |
| `redirect_uri` | console redirect page | Where TrueLayer sends the user after consent |
| `providers` | `uk-ob-all uk-oauth-all` | Which mock banks to show |

### Build the auth URL

Complete the auth builder in the TrueLayer Console


---
## Extract the authorisation code

After approving consent your browser is redirected to:
```
https://console.truelayer.com/redirect-page?code=XXXXXXXX&scope=...
```

### Parse the redirect URL

Paste your full redirect URL below and extract the `code` value.


In [ ]:
auth_code = ""


---
## Exchange the code for an access token

The auth code is single-use and expires in ~10 minutes.
Exchange it for an `access_token` via a POST to the token endpoint.

```
POST https://auth.truelayer-sandbox.com/connect/token
Content-Type: application/x-www-form-urlencoded
```

### Token exchange

Make the POST request and extract `access_token` and `refresh_token`.


In [ ]:
# Exercise 3.1 — token exchange
url = f"{AUTH_BASE}/connect/token"

payload = {
    "grant_type"    : "authorization_code",
    "client_id"     : CLIENT_ID,
    "client_secret" : CLIENT_SECRET,
    "code"          : auth_code,
    "redirect_uri"  : REDIRECT_URI,
}

response = requests.post(url, data=payload)

if response.status_code != 200:
    print(f"Token exchange failed: {response.status_code}")
    print(response.json())
else:
    token_data    = response.json()
    access_token  = token_data["access_token"]
    refresh_token = token_data["refresh_token"]
    expires_in    = token_data.get("expires_in", "unknown")

    print(f"Token exchange successful")
    print("  access_token  : [redacted]")
    print("  refresh_token : [redacted]")
    print(f"  expires_in    : {expires_in}s ({int(expires_in)//60} min)")
    print()
    print("Note: access tokens expire after ~1 hour.")
    print("Use the refresh_token to get a new one without re-doing consent.")


### Build a reusable API helper

All subsequent calls follow the same pattern: `GET {endpoint}` with `Authorization: Bearer {access_token}`.
Write a helper so we don't repeat the boilerplate.

The function should:
- Accept an endpoint path (e.g. `/accounts`)
- Add the Bearer token header automatically
- Raise a clear error if the status code is not 200
- Return the parsed JSON response


In [ ]:
def tl_get(endpoint):
    url     = f"{DATA_BASE}{endpoint}"
    headers = {"Authorization": f"Bearer {access_token}"}
    resp    = requests.get(url, headers=headers)

    if resp.status_code != 200:
        raise RuntimeError(
            f"API error {resp.status_code} on {endpoint}:\n{resp.text}"
        )
    return resp.json()

print("tl_get() defined — testing with /accounts...")
test = tl_get("/accounts")
print(f"  Status: OK — {len(test.get('results', []))} account(s) found")


---
## Accounts endpoint

```
GET /data/v1/accounts
```

Returns every bank account the user consented to share.
Each account has an `account_id` you use for all subsequent calls.

### Fetch and inspect accounts

Call `/accounts`, print the raw JSON for the first result, then load all accounts into a DataFrame.


In [ ]:
accounts_raw = tl_get("/accounts")

print("Raw JSON — first account:")
print(json.dumps(accounts_raw["results"][0], indent=2))

df_accounts = pd.DataFrame(accounts_raw["results"])
print(f"\n{len(df_accounts)} account(s) loaded")
print(df_accounts[["account_id", "account_type", "display_name", "currency"]].to_string(index=False))


### Understand the account object

Look at the raw JSON above. Answer the following in this cell:

1. What is the `account_type`? What other types might you see in a real dataset?
2. What does `provider` tell you, and why is it useful when aggregating across multiple banks?
3. Why does TrueLayer nest `account_number.number` and `account_number.sort_code` inside an object?



**1. account_type** — Sandbox typically returns `TRANSACTION` (current account).
In production you also see `SAVINGS` and `CARD` (credit card).
The type determines which endpoints apply: only `TRANSACTION`/`SAVINGS` work with `/balance`;
credit cards use `/cards` instead.

**2. provider** — Contains `display_name`, `logo_uri`, and `provider_id` (e.g. `ob-monzo`).
Critical for multi-bank aggregation: lets you attribute data to the right institution,
apply institution-specific parsing rules, and display the correct bank logo in a UI.

**3. Nested account_number** — Bank identifiers are not globally standardised.
UK accounts use sort code + account number; SEPA uses IBANs; US uses routing + account number.
Nesting with a `number_type` field supports all formats without nullable columns — a classic *discriminated union* API pattern.


---
## Balance endpoint

```
GET /data/v1/accounts/{account_id}/balance
```

Returns `current` and `available` balance for a specific account.

### Fetch balances for all accounts

Loop over every account and fetch its balance.
Build a DataFrame with `account_id`, `current`, `available`, `currency`.


In [ ]:
balance_rows = []

for account_id in df_accounts["account_id"]:
    resp   = tl_get(f"/accounts/{account_id}/balance")
    result = resp["results"][0]
    balance_rows.append({
        "account_id" : account_id,
        "current"    : result["current"],
        "available"  : result["available"],
        "currency"   : result["currency"],
        "updated"    : result.get("update_timestamp"),
    })

df_balances = pd.DataFrame(balance_rows)
print(f"Balances fetched for {len(df_balances)} account(s)")
print(df_balances.to_string(index=False))
print()
print("'current' = all posted transactions included.")
print("'available' = current minus pending holds or overdraft limits.")


### Current vs available balance

Calculate the difference between `current` and `available` for each account.
What could explain a gap between the two? Write your reasoning as a comment.


In [ ]:
df_balances["held_funds"] = df_balances["current"] - df_balances["available"]

print(df_balances[["account_id", "current", "available", "held_funds"]].to_string(index=False))

print("""
held_funds > 0 means the bank is holding funds not yet available to spend.
Common causes:
  - Card authorisations approved but not yet cleared (hotel pre-auth, petrol hold)
  - Direct debit notified but not yet debited
  - Cheque deposited but clearing period incomplete

This is the auth vs clearing distinction:
an authorised-but-not-cleared transaction reduces 'available' but not 'current'.
""")


---
## Transactions endpoint

```
GET /data/v1/accounts/{account_id}/transactions
```

Returns **cleared** transactions — the issuer's settled view, not real-time auth data.

### Inspect the raw response

Fetch transactions for your first account.
Pretty-print the first two transaction objects and identify which fields are always present vs sometimes missing.


In [ ]:
first_account_id = df_accounts["account_id"].iloc[0]
txn_raw          = tl_get(f"/accounts/{first_account_id}/transactions")
results          = txn_raw["results"]

print(f"Total transactions returned: {len(results)}")
print()
print("First transaction:")
print(json.dumps(results[0], indent=2))
print()
print("Second transaction:")
print(json.dumps(results[1], indent=2))


### Fetch transactions across all accounts

Loop over all accounts and collect every transaction into a single flat DataFrame.
Tag each row with its `account_id`.

Expected columns: `transaction_id`, `timestamp`, `description`, `amount`, `currency`,
`transaction_type`, `transaction_category`, `merchant_name`, `account_id`.


In [ ]:
all_transactions = []

for account_id in df_accounts["account_id"]:
    resp    = tl_get(f"/accounts/{account_id}/transactions")
    results = resp["results"]

    for txn in results:
        all_transactions.append({
            "transaction_id"       : txn.get("transaction_id"),
            "timestamp"            : txn.get("timestamp"),
            "description"          : txn.get("description"),
            "amount"               : txn.get("amount"),
            "currency"             : txn.get("currency"),
            "transaction_type"     : txn.get("transaction_type"),
            "transaction_category" : txn.get("transaction_category"),
            "merchant_name"        : txn.get("merchant_name"),
            "account_id"           : account_id,
        })

df_txn = pd.DataFrame(all_transactions)
df_txn["timestamp"] = pd.to_datetime(df_txn["timestamp"])

print(f"{len(df_txn)} transactions across {df_txn['account_id'].nunique()} account(s)")
print(df_txn.dtypes)
print()
print(df_txn.head(5).to_string())


### Data quality check

Before analysis, always profile your data. For `df_txn`:

1. Count missing values per column
2. Check for duplicate `transaction_id`s
3. Print the distribution of `transaction_type` and `transaction_category`
4. Show the min and max `timestamp` — what date range is covered?


In [ ]:
print("── 1. Missing values ─────────────────────")
print(df_txn.isnull().sum())

print("\n── 2. Duplicate transaction_ids ──────────")
n_dupes = df_txn["transaction_id"].duplicated().sum()
print(f"Duplicates: {n_dupes}")
if n_dupes > 0:
    print("Investigate — could be retries or API pagination overlap")

print("\n── 3a. transaction_type ──────────────────")
print(df_txn["transaction_type"].value_counts())

print("\n── 3b. transaction_category ──────────────")
print(df_txn["transaction_category"].value_counts())

print("\n── 4. Date range ─────────────────────────")
print(f"Earliest     : {df_txn['timestamp'].min()}")
print(f"Latest       : {df_txn['timestamp'].max()}")
print(f"Days covered : {(df_txn['timestamp'].max() - df_txn['timestamp'].min()).days}")

print("""
Common findings:
  - merchant_name is often null — not all banks enrich this field
  - transaction_category varies by bank — not a reliable ground truth
  - description is always present but inconsistent across institutions
  - Sandbox typically covers 3–6 months of mock history
""")


### Standing orders and direct debits

TrueLayer exposes two additional account-level endpoints that are critical for
understanding recurring payment behaviour — important context in any open banking
payment product.

```
GET /data/v1/accounts/{account_id}/standing_orders
GET /data/v1/accounts/{account_id}/direct_debits
```

Fetch both for your first account and answer:
1. What fields distinguish a standing order from a direct debit?
2. What would you use these for in a payment risk or affordability model?


In [ ]:
first_account_id = df_accounts["account_id"].iloc[0]

# Standing orders
so_raw = tl_get(f"/accounts/{first_account_id}/standing_orders")
print(f"Standing orders: {len(so_raw['results'])}")
if so_raw["results"]:
    print(json.dumps(so_raw["results"][0], indent=2))
df_so = pd.DataFrame(so_raw["results"])

# Direct debits
dd_raw = tl_get(f"/accounts/{first_account_id}/direct_debits")
print(f"\nDirect debits: {len(dd_raw['results'])}")
if dd_raw["results"]:
    print(json.dumps(dd_raw["results"][0], indent=2))
df_dd = pd.DataFrame(dd_raw["results"])

print("""
Key differences:
  - Standing order: payer-initiated, fixed amount, fixed frequency (push payment)
  - Direct debit: payee-initiated, variable amount, variable timing (pull payment)

Use cases in payment / affordability models:
  - Standing orders signal committed outgoings (rent, loan repayments)
  - Direct debits reveal utility/subscription obligations
  - Both reduce 'free cash flow' for affordability scoring
  - High DD count with variable amounts = more financial complexity / risk
""")


---
## Cards endpoint

```
GET /data/v1/cards
GET /data/v1/cards/{account_id}/balance
GET /data/v1/cards/{account_id}/transactions
```

Credit cards use a **separate** `/cards` endpoint.
The balance model differs: `credit_limit`, `last_statement_balance`, `payment_due`
instead of `current`/`available`.

### Fetch card accounts

Call `/cards` and inspect the response. How does a card account differ from a bank account?


In [ ]:
cards_raw = tl_get("/cards")

if not cards_raw["results"]:
    print("No card accounts in this sandbox session.")
    print("Re-authenticate and select a sandbox bank that includes a credit card.")
    df_cards = pd.DataFrame()
else:
    print("Raw JSON — first card:")
    print(json.dumps(cards_raw["results"][0], indent=2))

    df_cards = pd.DataFrame(cards_raw["results"])
    print(f"\n{len(df_cards)} card account(s) found")
    print(df_cards[["account_id", "display_name", "card_type", "card_network"]].to_string(index=False))


### Fetch card balance and transactions

For each card, fetch the balance and transactions.
Combine with bank transactions into a single unified DataFrame with a `source` column (`account` or `card`).


In [ ]:
card_transactions = []

if not df_cards.empty:
    for card_id in df_cards["account_id"]:
        bal  = tl_get(f"/cards/{card_id}/balance")["results"][0]
        print(f"Card {card_id[:12]}... | "
              f"Limit: {bal.get('currency')} {bal.get('credit_limit')} | "
              f"Available: {bal.get('available')} | "
              f"Statement bal: {bal.get('last_statement_balance')} | "
              f"Due: {bal.get('payment_due')} on {bal.get('payment_due_date')}")

        txn_resp = tl_get(f"/cards/{card_id}/transactions")
        for txn in txn_resp["results"]:
            card_transactions.append({
                "transaction_id"       : txn.get("transaction_id"),
                "timestamp"            : txn.get("timestamp"),
                "description"          : txn.get("description"),
                "amount"               : txn.get("amount"),
                "currency"             : txn.get("currency"),
                "transaction_type"     : txn.get("transaction_type"),
                "transaction_category" : txn.get("transaction_category"),
                "merchant_name"        : txn.get("merchant_name"),
                "account_id"           : card_id,
                "source"               : "card",
            })

df_txn_tagged = df_txn.copy()
df_txn_tagged["source"] = "account"

df_cards_txn = pd.DataFrame(card_transactions)
if not df_cards_txn.empty:
    df_cards_txn["timestamp"] = pd.to_datetime(df_cards_txn["timestamp"])

df_all = pd.concat([df_txn_tagged, df_cards_txn], ignore_index=True)
print(f"\nCombined dataset: {len(df_all)} transactions")
print(df_all["source"].value_counts())


---
## Clean and export

### Apply cleaning steps

Transform `df_all` and document each decision:

1. Drop exact duplicate `transaction_id`s (keep first)
2. Strip and lowercase `description`
3. Create `signed_amount`: negative for DEBIT, positive for CREDIT
4. Extract `date` (date only) and `hour` from `timestamp`
5. Fill missing `transaction_category` with `"UNKNOWN"`


In [ ]:
df_clean = df_all.copy()

before = len(df_clean)
df_clean = df_clean.drop_duplicates(subset="transaction_id", keep="first")
print(f"Step 1 — dropped {before - len(df_clean)} duplicate transaction_ids")

df_clean["description"] = df_clean["description"].str.strip().str.lower()
print("Step 2 — description normalised")

df_clean["signed_amount"] = df_clean.apply(
    lambda r: -abs(r["amount"]) if r["transaction_type"] == "DEBIT" else abs(r["amount"]),
    axis=1
)
print("Step 3 — signed_amount created")

df_clean["date"] = df_clean["timestamp"].dt.date
df_clean["hour"] = df_clean["timestamp"].dt.hour
print("Step 4 — date and hour extracted")

n_missing = df_clean["transaction_category"].isna().sum()
df_clean["transaction_category"] = df_clean["transaction_category"].fillna("UNKNOWN")
print(f"Step 5 — filled {n_missing} missing transaction_category values")

print(f"\nFinal clean shape: {df_clean.shape}")
print(df_clean.head(3).to_string())


### Basic summary statistics

Before saving, generate a brief summary:

- Total credits and total debits (using `signed_amount`)
- Top 5 transaction categories by count
- Top 5 merchants by total spend (debits only)


In [ ]:
total_credits = df_clean[df_clean["signed_amount"] > 0]["signed_amount"].sum()
total_debits  = df_clean[df_clean["signed_amount"] < 0]["signed_amount"].sum()
net           = total_credits + total_debits

currency = df_clean["currency"].iloc[0]
print(f"Total credits : +{total_credits:,.2f} {currency}")
print(f"Total debits  :  {total_debits:,.2f} {currency}")
print(f"Net           :  {net:,.2f} {currency}")

print("\nTop 5 categories by transaction count:")
print(df_clean["transaction_category"].value_counts().head(5).to_string())

print("\nTop 5 merchants by total spend (debits only):")
debits = df_clean[df_clean["signed_amount"] < 0].copy()
debits["abs_amount"] = debits["signed_amount"].abs()
top_merchants = (
    debits[debits["merchant_name"].notna()]
    .groupby("merchant_name")["abs_amount"]
    .sum()
    .sort_values(ascending=False)
    .head(5)
)

### Export to CSV

Save `df_clean` to `transactions_lesson1.csv` with a metadata header.


In [ ]:
STUDENT_NAME = "YOUR_NAME"
filename     = f"transactions_lesson1_{STUDENT_NAME.lower().replace(' ', '_')}.csv"

df_clean.to_csv(filename, index=False)

print(f"Saved: {filename}")
print(f"  Rows    : {len(df_clean)}")
print(f"  Columns : {list(df_clean.columns)}")
print(f"  Exported: {datetime.now().strftime('%Y-%m-%d %H:%M')}")
print()
print("This file is your carry-forward dataset for Lesson 2.")


---
## Token refresh (bonus)

Access tokens expire after ~1 hour. The `offline_access` scope we requested
gives us a `refresh_token` to get a new access token without repeating the
consent flow.

### Implement token refresh

Write a `tl_refresh()` function that:
- Sends the refresh token to the token endpoint
- Updates the global `access_token` variable
- Prints the new expiry time


In [ ]:
def tl_refresh():
    global access_token
    url = f"{AUTH_BASE}/connect/token"
    payload = {
        "grant_type"    : "refresh_token",
        "client_id"     : CLIENT_ID,
        "client_secret" : CLIENT_SECRET,
        "refresh_token" : refresh_token,
    }
    resp = requests.post(url, data=payload)
    if resp.status_code != 200:
        raise RuntimeError(f"Refresh failed: {resp.status_code}\n{resp.text}")

    data         = resp.json()
    access_token = data["access_token"]
    expires_in   = data.get("expires_in", 3600)
    print(f"Token refreshed — expires in {expires_in}s ({expires_in//60} min)")
    return access_token

# tl_refresh()  # uncomment to test


---
## Payment analytics exercises

These exercises connect the raw AIS data to the kind of analysis done
in a real open banking payment product. They are progressively harder.

---

### Monthly spend trend

Calculate total debit spend per month. Plot a bar chart (or print the table).
Identify the highest and lowest spend months and hypothesise why they differ.


In [ ]:
monthly = (
    df_clean[df_clean["signed_amount"] < 0]
    .assign(month=df_clean["timestamp"].dt.to_period("M"))
    .groupby("month")["signed_amount"]
    .sum()
    .abs()
    .reset_index()
    .rename(columns={"signed_amount": "total_debit"})
    .sort_values("month")
)

print("Monthly debit spend:")
print(monthly.to_string(index=False))

peak   = monthly.loc[monthly["total_debit"].idxmax()]
trough = monthly.loc[monthly["total_debit"].idxmin()]
print(f"\nHighest spend month : {peak['month']} — {peak['total_debit']:,.2f}")
print(f"Lowest  spend month : {trough['month']} — {trough['total_debit']:,.2f}")


### Category breakdown

For each `transaction_category`, calculate:
- Transaction count
- Total spend
- Average transaction size
- % of total spend

Sort by total spend descending.


In [ ]:
debits = df_clean[df_clean["signed_amount"] < 0].copy()
debits["abs_amount"] = debits["signed_amount"].abs()
total_spend = debits["abs_amount"].sum()

cat_summary = (
    debits.groupby("transaction_category")
    .agg(
        count        = ("transaction_id", "count"),
        total_spend  = ("abs_amount", "sum"),
        avg_amount   = ("abs_amount", "mean"),
    )
    .assign(pct_of_total=lambda df: (df["total_spend"] / total_spend * 100).round(1))
    .sort_values("total_spend", ascending=False)
)

print("Category breakdown (debits only):")
print(cat_summary.to_string())


### Hourly transaction patterns

Aggregate transactions by hour of day. Separate DEBIT and CREDIT.
What does the hourly distribution tell you about user payment behaviour?
When would you expect peaks for a pay-by-bank product?


### Recurring payment detection

Identify likely **recurring** transactions: same description, same amount (±5%),
appearing at roughly monthly intervals.

This is a simplified version of what open banking affordability engines use to
identify committed spend commitments.


In [ ]:
import numpy as np

df_rec = df_clean[df_clean["signed_amount"] < 0].copy()
df_rec["abs_amount"]    = df_rec["signed_amount"].abs()
df_rec["amount_bucket"] = (df_rec["abs_amount"] / 5).round() * 5   # round to nearest 5
df_rec["month"]         = df_rec["timestamp"].dt.to_period("M")

recurring = (
    df_rec.groupby(["description", "amount_bucket"])
    .agg(
        month_count  = ("month",          pd.Series.nunique),
        total_txns   = ("transaction_id", "count"),
        avg_amount   = ("abs_amount",     "mean"),
    )
    .query("month_count >= 2")
    .sort_values("month_count", ascending=False)
)

print(f"Likely recurring transactions ({len(recurring)} patterns):")
print(recurring.head(15).to_string())

print("""
In a production affordability model:
  - Recurring debits are treated as committed outgoings
  - They reduce disposable income available for new credit obligations
  - Missed recurrings (pattern stops) are a leading indicator of financial stress
  - Direct debits + standing orders + recurring card patterns = full committed spend picture
""")


### First-transaction funnel (new user simulation)

Imagine each sandbox account represents a new user's first 30 days with a
pay-by-bank product. Using only the first 30 days of transaction history per account:

1. How many transactions did each account make in day 1 vs day 7 vs day 30?
2. What % of accounts had at least one DEBIT in the first 7 days?
3. Build a simple day-by-day cumulative spend curve per account.

This mirrors the new-user funnel analysis done for iGaming clients.


In [ ]:
df_funnel = df_clean.copy()

# Day 0 = first transaction date per account
first_txn = df_funnel.groupby("account_id")["timestamp"].min().rename("first_txn")
df_funnel  = df_funnel.merge(first_txn, on="account_id")
df_funnel["days_since_first"] = (df_funnel["timestamp"] - df_funnel["first_txn"]).dt.days

# Limit to first 30 days
df_30 = df_funnel[df_funnel["days_since_first"] <= 30]

# 1. Transaction count at day 1, 7, 30
for d in [1, 7, 30]:
    n = df_30[df_30["days_since_first"] <= d].groupby("account_id").size()
    print(f"Avg transactions by day {d:2d}: {n.mean():.1f} | median: {n.median():.1f}")

# 2. % accounts with at least one DEBIT in first 7 days
week1_debit = (
    df_30[(df_30["days_since_first"] <= 7) & (df_30["transaction_type"] == "DEBIT")]
    ["account_id"].nunique()
)
total_accounts = df_30["account_id"].nunique()
print(f"\n% accounts with DEBIT in first 7 days: {week1_debit/total_accounts*100:.1f}%")

# 3. Cumulative spend curve
cumulative = (
    df_30[df_30["signed_amount"] < 0]
    .sort_values("days_since_first")
    .groupby(["account_id", "days_since_first"])["signed_amount"]
    .sum()
    .abs()
    .groupby(level=0)
    .cumsum()
    .reset_index()
    .groupby("days_since_first")["signed_amount"]
    .mean()
)

print("\nAverage cumulative spend by day (all accounts):")
print(cumulative.head(31).to_string())


---
Pagination and large datasets (advanced)

The TrueLayer sandbox returns all transactions in a single response.
In production, endpoints paginate using `from` and `to` date range parameters:

```
GET /data/v1/accounts/{id}/transactions?from=2024-01-01T00:00:00Z&to=2024-01-31T23:59:59Z
```


---
## Summary

Full pipeline completed:

| Step | What you did | Key concept |
|---|---|---|
| 1–3 | OAuth2 consent flow | Auth code → token exchange |
| 4 | `/accounts` | Account discovery, provider metadata |
| 5 | `/accounts/{id}/balance` | Current vs available balance |
| 6 | `/accounts/{id}/transactions` | Cleared transaction data, DQ checks |
| 6.4 | `/standing_orders` + `/direct_debits` | Recurring committed spend |
| 7 | `/cards` endpoints | Credit card vs bank data model |
| 8 | Cleaning + export | Reproducible pipeline |
| 9 | Token refresh | Long-lived integrations |
| 10 | Analytics exercises | Monthly trends, HHI, recurring detection, user funnel |
| 11 | Paginated fetch | Production-scale data extraction |

### Things to remember for Lesson 2

- `description` is raw bank text — inconsistent across institutions. Lesson 2 uses NLP to categorise it.
- `merchant_name` is an enriched field, absent for many transactions.
- `transaction_category` from TrueLayer is the bank's own classification — not a reliable ground truth.
- You are seeing **cleared transactions** only — no auth-level data in AIS.
- Standing orders + direct debits + recurring card patterns together give you committed spend.
